# Pools silver layer

### Imports

In [0]:
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Execution

In [0]:
bz_pools_df = spark.read.table(f"uniswap.bronze.{entity_name}")


In [0]:
pools_df = bz_pools_df.withColumn("load_timestamp", current_timestamp())

In [0]:
if load_pattern == "1":
    print("*INFO*: Executing full load.")
    pools_df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"uniswap.silver.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "2":
    print("*INFO*: Executing incremental load.")
    pools_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"uniswap.silver.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "3":
    print("*INFO*: Executing differential load.")
    # DeltaTables merge
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")
else:
    print("*ERROR*: Invalid load pattern.")
    dbutils.notebook.exit("Invalid load pattern.")